In [6]:
# Importations TensorFlow/Keras, SciKit-Learn, PIL, OS, NumPy, OpenCV, Matplotlib
from google.colab import drive
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.metrics import Recall, AUC # Métriques Keras intégrées
from sklearn.model_selection import train_test_split
from PIL import Image
import os
import numpy as np
import random
import cv2
import matplotlib.pyplot as plt
import time # Pour mesurer le temps

# Importations pour métriques post-entraînement
from scipy.spatial.distance import directed_hausdorff # Pour Hausdorff

# Montage Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Fonctions d'augmentation (rotateImage, bruit, change_gamma, color, random_change)

def rotateImage(image, angle):
    image_center = tuple(np.array(image.shape[1::-1])/2)
    rot_mat = cv2.getRotationMatrix2D(image_center, angle, 1.0)
    # Correction: Utiliser image.shape[1::-1] pour les dimensions de sortie de warpAffine
    return cv2.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv2.INTER_LINEAR)

def bruit(image):
    # Ajoute du bruit gaussien aléatoire
    # Assurez-vous que l'image est en float pour l'addition avant clip/astype
    image_float = image.astype(np.float32)
    noise = np.random.randn(*image.shape) * random.randint(5, 30)
    noisy_image = np.clip(image_float + noise, 0, 255)
    return noisy_image.astype(np.uint8)

def change_gamma(image, alpha=1.0, beta=0.0):
    # Ajuste la luminosité (beta) et le contraste (alpha)
    # Assurez-vous que l'image est en float pour la multiplication/addition
    image_float = image.astype(np.float32)
    adjusted_image = np.clip(alpha * image_float + beta, 0, 255)
    return adjusted_image.astype(np.uint8)

def color(image, alpha=20):
    # Ajoute une variation de couleur aléatoire
    # Conversion en int32 pour éviter l'overflow/underflow pendant l'addition
    variation = np.random.randint(-alpha, alpha + 1, size=image.shape, dtype=np.int32)
    colored_image = np.clip(image.astype(np.int32) + variation, 0, 255)
    return colored_image.astype(np.uint8)

def random_change(image):
    # Applique aléatoirement certaines transformations (comme votre code)
    img = image.copy() # Travaille sur une copie
    if np.random.randint(2):
        img = change_gamma(img, random.uniform(0.8, 1.2), np.random.randint(-50, 50))
    if np.random.randint(2):
        img = bruit(img)
    if np.random.randint(2):
        img = color(img)
    return img

In [9]:
# --- Fonction Dice Coefficient  ---
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

# --- Modèle U-Net (Votre fonction 'model' originale) ---
def model(nbr):
    entree = layers.Input(shape=(576, 560, 3), dtype='float32')

    # Encodeur
    conv1 = layers.Conv2D(nbr, 3, activation='relu', padding='same')(entree)
    conv1 = layers.BatchNormalization()(conv1)
    conv1 = layers.Conv2D(nbr, 3, activation='relu', padding='same')(conv1)
    bn1 = layers.BatchNormalization()(conv1)
    pool1 = layers.MaxPool2D()(bn1)

    conv2 = layers.Conv2D(2 * nbr, 3, activation='relu', padding='same')(pool1)
    conv2 = layers.BatchNormalization()(conv2)
    conv2 = layers.Conv2D(2 * nbr, 3, activation='relu', padding='same')(conv2)
    bn2 = layers.BatchNormalization()(conv2)
    pool2 = layers.MaxPool2D()(bn2)

    conv3 = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(pool2)
    conv3 = layers.BatchNormalization()(conv3)
    conv3 = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(conv3)
    bn3 = layers.BatchNormalization()(conv3)
    pool3 = layers.MaxPool2D()(bn3)

    conv4a = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(pool3) # Renommé pour éviter confusion avec décodeur
    conv4a = layers.BatchNormalization()(conv4a)
    conv4a = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(conv4a)
    bn4 = layers.BatchNormalization()(conv4a)
    pool4 = layers.MaxPool2D()(bn4)

    # Bottleneck (8*nbr puis 4*nbr)
    conv_mid = layers.Conv2D(8 * nbr, 3, activation='relu', padding='same')(pool4)
    conv_mid = layers.BatchNormalization()(conv_mid)
    conv_mid = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(conv_mid)
    bn_mid = layers.BatchNormalization()(conv_mid)

    # Décodeur
    up1 = layers.UpSampling2D()(bn_mid)
    concat1 = layers.Concatenate(axis=3)([up1, bn4]) # Skip connection depuis bn4
    # Blocs Conv après concat (8*nbr puis 4*nbr )
    dconv1 = layers.Conv2D(8 * nbr, 3, activation='relu', padding='same')(concat1)
    dconv1 = layers.BatchNormalization()(dconv1)
    dconv1 = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(dconv1)
    dbn1 = layers.BatchNormalization()(dconv1)

    up2 = layers.UpSampling2D()(dbn1)
    concat2 = layers.Concatenate(axis=3)([up2, bn3]) # Skip connection depuis bn3
    # Blocs Conv après concat (4*nbr puis 2*nbr )
    dconv2 = layers.Conv2D(4 * nbr, 3, activation='relu', padding='same')(concat2)
    dconv2 = layers.BatchNormalization()(dconv2)
    dconv2 = layers.Conv2D(2 * nbr, 3, activation='relu', padding='same')(dconv2)
    dbn2 = layers.BatchNormalization()(dconv2)

    up3 = layers.UpSampling2D()(dbn2)
    concat3 = layers.Concatenate(axis=3)([up3, bn2]) # Skip connection depuis bn2
    # Blocs Conv après concat (2*nbr puis nbr )
    dconv3 = layers.Conv2D(2 * nbr, 3, activation='relu', padding='same')(concat3)
    dconv3 = layers.BatchNormalization()(dconv3)
    dconv3 = layers.Conv2D(nbr, 3, activation='relu', padding='same')(dconv3)
    dbn3 = layers.BatchNormalization()(dconv3)

    up4 = layers.UpSampling2D()(dbn3)
    concat4 = layers.Concatenate(axis=3)([up4, bn1]) # Skip connection depuis bn1
    # Blocs Conv après concat (nbr puis nbr)
    dconv4 = layers.Conv2D(nbr, 3, activation='relu', padding='same')(concat4)
    dconv4 = layers.BatchNormalization()(dconv4)
    dconv4 = layers.Conv2D(nbr, 3, activation='relu', padding='same')(dconv4)
    dbn4 = layers.BatchNormalization()(dconv4)

    # Couche de sortie (comme votre code)
    sortie = layers.Conv2D(1, 1, activation='sigmoid', padding='same')(dbn4)

    return models.Model(inputs=entree, outputs=sortie)


# --- Nouvelles métriques Keras personnalisées (Inchangées) ---

def iou_coef(y_true, y_pred, smooth=1e-6):
    """Coefficient IoU (Intersection over Union) ou Indice de Jaccard."""
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return iou

# Sensitivity est géré par tf.keras.metrics.Recall(name='sensitivity')

def specificity(y_true, y_pred):
    """Spécificité (True Negative Rate)."""
    neg_y_true = 1 - y_true
    neg_y_pred = 1 - y_pred
    fp = K.sum(K.cast(K.greater(y_pred, 0.5), 'float32') * K.cast(K.equal(y_true, 0), 'float32'))
    tn = K.sum(K.cast(K.less_equal(y_pred, 0.5), 'float32') * K.cast(K.equal(y_true, 0), 'float32'))
    specificity_val = tn / (tn + fp + K.epsilon())
    return specificity_val

In [12]:
# Chemins vers les données
dir_images = '/content/drive/MyDrive/DRIVE/training/images/'
dir_mask   = '/content/drive/MyDrive/DRIVE/training/1st_manual/'

# Vérification existence dossiers
if not os.path.isdir(dir_images):
    raise ValueError(f"Le dossier d'images n'existe pas : {dir_images}")
if not os.path.isdir(dir_mask):
    raise ValueError(f"Le dossier de masques n'existe pas : {dir_mask}")

tab_images, tab_masks = [], []
target_height, target_width = 576, 560

print("Chargement et augmentation des données...")

# Boucle de chargement et d'augmentation
for fichier in sorted(os.listdir(dir_images)):
    if fichier.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
        try:
            img_path = os.path.join(dir_images, fichier)
            img_orig = cv2.imread(img_path)
            if img_orig is None: continue
            img_orig = img_orig[:target_height, :target_width]
            if img_orig.shape[0] != target_height or img_orig.shape[1] != target_width:
                img_orig = cv2.resize(img_orig, (target_width, target_height), interpolation=cv2.INTER_AREA)

            num = fichier.split('_')[0]
            file_mask = os.path.join(dir_mask, num + '_manual1.gif')
            if not os.path.exists(file_mask): continue

            img_mask_orig = np.array(Image.open(file_mask))
            if len(img_mask_orig.shape) == 3: img_mask_orig = img_mask_orig[:,:,0]
            img_mask_orig = img_mask_orig[:target_height, :target_width]
            if img_mask_orig.shape[0] != target_height or img_mask_orig.shape[1] != target_width:
                 img_mask_orig = cv2.resize(img_mask_orig, (target_width, target_height), interpolation=cv2.INTER_NEAREST)

            tab_images.append(img_orig)
            tab_masks.append(img_mask_orig)

            for angle in [0, 90, 180, 270]:
                img_r = rotateImage(img_orig, angle)
                mask_rot_mat = cv2.getRotationMatrix2D(tuple(np.array(img_mask_orig.shape[1::-1])/2), angle, 1.0)
                img_mask_r = cv2.warpAffine(img_mask_orig, mask_rot_mat, img_mask_orig.shape[1::-1], flags=cv2.INTER_NEAREST)

                for flip_code in [-1, 0, 1]:
                    for flip in [0, 1]:
                        img_f = cv2.flip(img_r, flip)
                        img_mask_f = cv2.flip(img_mask_r, flip)

                        img_augmented = random_change(img_f)
                        img_augmented = img_augmented[:target_height, :target_width]
                        img_mask_final = img_mask_f[:target_height, :target_width]

                        if img_augmented.shape[0] == target_height and img_augmented.shape[1] == target_width and \
                           img_mask_final.shape[0] == target_height and img_mask_final.shape[1] == target_width:
                              tab_images.append(img_augmented)
                              tab_masks.append(img_mask_final)

        except Exception as e:
            print(f"Erreur pendant chargement/augmentation pour {fichier}: {e}")

if len(tab_images) > 500:
    print(f"Attention: Plus de 500 images générées ({len(tab_images)}), limitation à 500.")
    indices = random.sample(range(len(tab_images)), 500)
    tab_images = [tab_images[i] for i in indices]
    tab_masks = [tab_masks[i] for i in indices]

print(f"Chargement et augmentation terminés. Nombre total d'échantillons: {len(tab_images)}")
if not tab_images:
     raise SystemExit("Erreur: Aucune image/masque après chargement/augmentation.")

tab_images = np.array(tab_images, dtype=np.float32) / 255.0
tab_masks = np.array(tab_masks, dtype=np.float32) / 255.0
if len(tab_masks.shape) == 3:
    tab_masks = np.expand_dims(tab_masks, axis=-1)
    print(f"Dimension de canal ajoutée aux masques. Nouvelle forme : {tab_masks.shape}")

print(f"Forme finale des images : {tab_images.shape}")
print(f"Forme finale des masques : {tab_masks.shape}")

# Séparer les jeux de données (Création des ensembles d'entraînement, validation et test)
train_val_images, test_images, train_val_masks, test_masks = train_test_split(
    tab_images, tab_masks, test_size=0.05, random_state=42
)

train_images, val_images, train_masks, val_masks = train_test_split(
    train_val_images, train_val_masks, test_size=0.2, random_state=123
)

print(f"\nDonnées divisées :")
print(f"  Entraînement : {train_images.shape[0]} images, {train_masks.shape[0]} masques")
print(f"  Validation : {val_images.shape[0]} images, {val_masks.shape[0]} masques")
print(f"  Test : {test_images.shape[0]} images, {test_masks.shape[0]} masques")

# Libérer la mémoire
del tab_images, tab_masks, train_val_images, train_val_masks

Chargement et augmentation des données...
Chargement et augmentation terminés. Nombre total d'échantillons: 500
Dimension de canal ajoutée aux masques. Nouvelle forme : (500, 576, 560, 1)
Forme finale des images : (500, 576, 560, 3)
Forme finale des masques : (500, 576, 560, 1)

Données divisées :
  Entraînement : 380 images, 380 masques
  Validation : 95 images, 95 masques
  Test : 25 images, 25 masques


In [13]:
# Instance utilisée pour l'entraînement dans le code
my_model = model(64) # Crée le modèle U-Net standard avec 64 filtres de base

# Compiler le modèle avec les métriques ajoutées
my_model.compile(optimizer='adam',                 # optimiseur
                 loss='binary_crossentropy',       #  fonction de perte
                 metrics=['accuracy',                # Votre métrique originale
                          dice_coef,
                          iou_coef,                #  IoU / Jaccard
                          Recall(name='sensitivity'), #  Sensitivity (Recall)
                          specificity,             #  Specificity
                          AUC(name='auc_roc')      #  AUC-ROC
                          ])

# Afficher le résumé du modèle qui sera entraîné
print("\nRésumé du modèle U-Net qui sera entraîné :")
my_model.summary()


Résumé du modèle U-Net qui sera entraîné :


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 576, 560,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 576, 560,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 576, 560,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 576, 560,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 576, 560,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 288, 280,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 288, 280,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 288, 280,  │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 288, 280,  │    147,584 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 288, 280,  │        512 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 144, 140,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 144, 140,  │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 144, 140,  │      1,024 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 144, 140,  │    590,080 │ batch_normalizat… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 144, 140,  │      1,024 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 72, 70,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 72, 70,    │    590,080 │ max_pooling2d_2[

 Total params: 10,194,497 (38.89 MB)

 Trainable params: 10,187,201 (38.86 MB)

 Non-trainable params: 7,296 (28.50 KB)

In [1]:
# --- AJOUT DES CALLBACKS ---
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Définir le chemin pour sauvegarder le meilleur modèle
checkpoint_filepath_simple = '/content/drive/MyDrive/best_unet_simple.h5'

# Callback EarlyStopping: arrête l'entraînement si val_dice_coef ne s'améliore pas pendant 15 époques
# restore_best_weights=True garantit que le modèle aura les meilleurs poids à la fin
early_stopping = EarlyStopping(monitor='val_dice_coef', patience=15, verbose=1, mode='max', restore_best_weights=True)

# Callback ModelCheckpoint: sauvegarde le modèle uniquement lorsque val_dice_coef s'améliore
model_checkpoint = ModelCheckpoint(filepath=checkpoint_filepath_simple,
                                   save_weights_only=False, # Sauvegarde le modèle complet
                                   monitor='val_dice_coef',
                                   mode='max',
                                   save_best_only=True,
                                   verbose=1) # verbose=1 pour voir quand le modèle est sauvegardé

# --- PARAMETRES D'ENTRAINEMENT AJUSTES ---
# Mettre une valeur élevée, EarlyStopping s'occupera d'arrêter au bon moment
EPOCHS = 150
# BATCH_SIZE ajusté pour Colab T4 (compromis mémoire/stabilité) - À TESTER
# DOIT ÊTRE LA MÊME VALEUR que pour le notebook Attention U-Net
BATCH_SIZE = 4

print(f"\n--- Début de l'entraînement ({EPOCHS} époques max, batch size {BATCH_SIZE}) ---")
print(f"Early Stopping surveille '{early_stopping.monitor}' ({early_stopping.mode}) avec patience {early_stopping.patience}.")
print(f"Model Checkpoint sauvegardera le meilleur modèle selon '{model_checkpoint.monitor}' dans '{checkpoint_filepath_simple}'.")

start_training_time = time.time()

# Entraîner le modèle avec les callbacks
history = my_model.fit(train_images, train_masks,
                       epochs=EPOCHS,
                       batch_size=BATCH_SIZE,
                       validation_data=(val_images, val_masks), # UTILISATION DE L'ENSEMBLE DE VALIDATION ICI
                       callbacks=[early_stopping, model_checkpoint],
                       verbose=1) # verbose=1 pour voir la progression

end_training_time = time.time()
training_runtime = end_training_time - start_training_time
print(f"\n--- Entraînement terminé (potentiellement avant {EPOCHS} époques grâce à EarlyStopping) ---")
print(f"Temps d'exécution total de l'entraînement (Runtime) : {training_runtime:.2f} secondes ({training_runtime/60:.2f} minutes)")
print(f"Le meilleur modèle a été sauvegardé à : {checkpoint_filepath_simple}")
# Note: Grâce à restore_best_weights=True dans EarlyStopping, 'my_model' contient maintenant les meilleurs poids.


--- Début de l'entraînement (150 époques max, batch size 4) ---
Early Stopping surveille 'val_dice_coef' (max) avec patience 15.
Model Checkpoint sauvegardera le meilleur modèle selon 'val_dice_coef' dans '/content/drive/MyDrive/best_unet_simple.h5'.


NameError: name 'time' is not defined

In [ ]:
print("\n--- Génération des graphiques de l'historique d'entraînement ---")

history_dict = history.history

metrics_to_plot = {
    'loss': 'Loss',
    'accuracy': 'Accuracy',
    'dice_coef': 'Dice Coefficient',
    'iou_coef': 'IoU (Jaccard)',
    'sensitivity': 'Sensitivity (Recall)',
    'specificity': 'Specificity',
    'auc_roc': 'AUC-ROC'
}

num_metrics = len(metrics_to_plot)
num_cols = 3
num_rows = (num_metrics + num_cols - 1) // num_cols

plt.style.use('ggplot')
fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(num_cols * 6, num_rows * 4.5))
axes = axes.flatten()

plot_index = 0
for metric_key, metric_name in metrics_to_plot.items():
    if plot_index >= len(axes): break # Sécurité
    ax = axes[plot_index]
    if metric_key in history_dict:
        epochs_range = range(1, len(history_dict[metric_key]) + 1) # Pour l'axe X
        ax.plot(epochs_range, history_dict[metric_key], label=f'Train {metric_name}', marker='.')
        val_metric_key = f'val_{metric_key}'
        if val_metric_key in history_dict:
            ax.plot(epochs_range, history_dict[val_metric_key], label=f'Validation {metric_name}', marker='.')
        else:
            print(f"Note : Pas de données de validation trouvées pour '{val_metric_key}'")
        ax.set_title(f'{metric_name} vs. Epochs')
        ax.set_xlabel('Epochs')
        ax.set_ylabel(metric_name)
        ax.legend()
        plot_index += 1
    else:
        print(f"Attention : Métrique '{metric_key}' non trouvée dans l'historique.")

# Cacher axes non utilisés
for i in range(plot_index, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

# Affichage valeurs finales (ATTENTION: celles de la dernière époque, pas forcément les meilleures si EarlyStopping sans restore_best_weights)
# Si restore_best_weights=True, ces valeurs finales correspondent à la meilleure époque arrêtée
print("\n--- Valeurs finales des métriques (dernière époque effectuée / meilleure époque si restore_best_weights=True) ---")
final_epoch = len(history_dict.get('loss', []))
if final_epoch > 0:
  for metric_key, metric_name in metrics_to_plot.items():
      print(f"--- {metric_name} ---")
      train_val = history_dict.get(metric_key, [np.nan])[-1] # Prend la dernière valeur ou NaN
      val_val = history_dict.get(f'val_{metric_key}', [np.nan])[-1] # Prend la dernière valeur ou NaN
      print(f"  Train      : {train_val:.4f}")
      print(f"  Validation : {val_val:.4f}")
else:
  print("L'entraînement n'a pas produit d'historique.")

In [ ]:
print("\n--- Évaluation post-entraînement sur l'ensemble de validation ---")
# Note: Utilise test_images/test_masks qui ont servi de validation_data durant fit()
# 'my_model' a les meilleurs poids si restore_best_weights=True dans EarlyStopping

# 1. Mesurer le Temps d'Inférence (Inference Time) sur l'ensemble de validation
num_val_samples = len(test_images)
if num_val_samples > 0:
    print(f"Calcul du temps d'inférence sur {num_val_samples} images de validation...")
    start_inference_time = time.time()
    # Important: S'assurer que BATCH_SIZE ici est gérable pour l'inférence
    # Il peut être différent de celui de l'entraînement si besoin, mais doit être <= N images
    inference_batch_size = min(BATCH_SIZE, num_val_samples) # Utilise le BATCH_SIZE d'entraînement ou moins
    predictions_prob_val = my_model.predict(test_images, batch_size=inference_batch_size)
    end_inference_time = time.time()

    total_inference_time = end_inference_time - start_inference_time
    avg_inference_time_per_image = total_inference_time / num_val_samples
    print(f"  Temps d'inférence total pour {num_val_samples} images: {total_inference_time:.4f} secondes")
    print(f"  Temps d'inférence moyen par image: {avg_inference_time_per_image:.6f} secondes")

    # 2. Calculer la Distance de Hausdorff sur l'ensemble de validation
    print(f"\nCalcul de la distance de Hausdorff pour {num_val_samples} paires de masques...")
    predictions_binary_val = (predictions_prob_val > 0.5).astype(np.uint8)
    # Assurer que test_masks est aussi binaire pour comparaison Hausdorff
    test_masks_binary_val = (test_masks > 0.5).astype(np.uint8)

    hausdorff_distances = []
    for i in range(num_val_samples):
        gt_mask = test_masks_binary_val[i].squeeze()
        pred_mask = predictions_binary_val[i].squeeze()
        coords_gt = np.argwhere(gt_mask > 0)
        coords_pred = np.argwhere(pred_mask > 0)

        if coords_gt.shape[0] > 0 and coords_pred.shape[0] > 0:
            try:
                # Utilisation de scipy.spatial.distance.directed_hausdorff
                hd1 = directed_hausdorff(coords_gt, coords_pred)[0]
                hd2 = directed_hausdorff(coords_pred, coords_gt)[0]
                hausdorff_distances.append(max(hd1, hd2))
            except Exception as e:
                print(f" Avertissement: Erreur calcul Hausdorff pour l'image {i}: {e}")
                hausdorff_distances.append(np.nan) # En cas d'erreur (peu probable ici)
        elif coords_gt.shape[0] == 0 and coords_pred.shape[0] == 0:
            hausdorff_distances.append(0.0) # Si les deux sont vides, distance nulle
        else:
            # Si l'un est vide et l'autre non, la distance de Hausdorff n'est pas définie
            # ou peut être considérée comme infinie/maximale. On met NaN pour la moyenne.
            hausdorff_distances.append(np.nan)

    valid_distances = [d for d in hausdorff_distances if not np.isnan(d)]
    if valid_distances:
        avg_hausdorff = np.mean(valid_distances)
        std_hausdorff = np.std(valid_distances)
        print(f"\n  Distance de Hausdorff Moyenne (Validation Set, {len(valid_distances)} paires valides): {avg_hausdorff:.4f}")
        print(f"  Distance de Hausdorff Écart-type (Validation Set, {len(valid_distances)} paires valides): {std_hausdorff:.4f}")
        print("  (Note: La métrique 's2s distance' est souvent liée à Hausdorff ou ASD.)")
    else:
        print("\n  Aucune distance de Hausdorff valide n'a pu être calculée sur le set de validation.")

else:
    print("Ensemble de validation vide. Métriques post-entraînement non calculées.")

In [ ]:
print("\n--- Prédictions sur le dossier de test externe ---")

# Chemins et création dossier
dir_test_images = '/content/drive/MyDrive/DRIVE/test/images/'
dir_predictions = '/content/drive/MyDrive/DRIVE/NEW-predictions-unet-simple/' # Dossier spécifique
os.makedirs(dir_predictions, exist_ok=True)

# Vérification existence dossier test
if not os.path.isdir(dir_test_images):
    print(f"Warning: Dossier de test externe {dir_test_images} non trouvé. Section de prédiction ignorée.")
else:
    print(f"Chargement des images depuis: {dir_test_images}")
    print(f"Sauvegarde des prédictions dans: {dir_predictions}")

    # Chargement et prédiction
    fichiers_test = sorted(os.listdir(dir_test_images))
    processed_count = 0
    for fichier in fichiers_test:
         if fichier.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
            try:
                img_path = os.path.join(dir_test_images, fichier)
                image_ext = cv2.imread(img_path)
                if image_ext is None:
                    print(f"Warning: Impossible de lire {fichier}, ignoré.")
                    continue

                # Redimensionner/Recadrer à la taille attendue par le modèle
                h_orig, w_orig = image_ext.shape[:2]
                image_ext_resized = image_ext[:target_height, :target_width]
                if image_ext_resized.shape[0] != target_height or image_ext_resized.shape[1] != target_width:
                    # Redimensionne en gardant les proportions si possible, ou force la taille
                    image_ext_resized = cv2.resize(image_ext, (target_width, target_height), interpolation=cv2.INTER_AREA)


                # Normaliser et ajouter dimension batch
                image_ext_norm = np.array(image_ext_resized, dtype=np.float32) / 255.0
                image_ext_batch = image_ext_norm[np.newaxis, ...] # Ajoute la dimension batch (1, H, W, C)

                # Prédiction (utilise le modèle chargé avec les meilleurs poids si restore_best_weights=True)
                prediction_prob = my_model.predict(image_ext_batch)[0] # Prend la première prédiction

                # Redimensionner masque de sortie à 584x565 et sauvegarder (selon votre code original)
                # Note: Assurez-vous que cette taille de sortie est bien celle désirée
                output_h, output_w = 584, 565
                mask_out = np.zeros((output_h, output_w, 1), dtype=np.uint8)

                # Mettre à l'échelle la prédiction (0-1 -> 0-255)
                prediction_scaled = (prediction_prob * 255.0).astype(np.uint8)

                # S'assurer que la prédiction a la bonne dimension avant de la coller
                if prediction_scaled.shape[0] != target_height or prediction_scaled.shape[1] != target_width:
                     prediction_scaled_resized = cv2.resize(prediction_scaled, (target_width, target_height), interpolation=cv2.INTER_NEAREST)
                else:
                     prediction_scaled_resized = prediction_scaled

                # Ajouter la dimension canal si manquante après resize
                if len(prediction_scaled_resized.shape) == 2:
                    prediction_scaled_resized = np.expand_dims(prediction_scaled_resized, axis=-1)

                # Copier la prédiction (taille target_height x target_width) dans le coin du masque final (584x565)
                mask_out[:target_height, :target_width] = prediction_scaled_resized

                # Nom du fichier de sortie
                base_name = os.path.splitext(fichier)[0]
                save_path = os.path.join(dir_predictions, f"pred_{base_name}.png")

                cv2.imwrite(save_path, mask_out)
                processed_count += 1

            except Exception as e:
                print(f"Erreur lors de la prédiction pour {fichier}: {e}")

    print(f"{processed_count} prédictions sauvegardées dans {dir_predictions}")

In [ ]:
# Enregistrer l'état final du modèle (qui a les meilleurs poids si restore_best_weights=True)
# Peut être différent du checkpoint si l'entraînement s'est poursuivi après le meilleur score
model_save_path_final = '/content/drive/MyDrive/final_unet_simple.h5'
my_model.save(model_save_path_final)
print(f"\nModèle final (U-Net Simple) enregistré sous : {model_save_path_final}")
print(f"Le MEILLEUR modèle basé sur '{model_checkpoint.monitor}' est dans : {checkpoint_filepath_simple}")

print("\n--- Fin du script U-Net Simple ---")